Import the necessary libraries for prediction

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
# import scikeras
# from scikeras.wrappers import KerasRegressor
from math import sqrt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import explained_variance_score
from sklearn.metrics import r2_score

Load data and check output

In [ ]:
df=pd.read_csv("G:/HH_1/Table_Point_1/1.csv",parse_dates=["DATE"],index_col=[0])
df.head()

In [ ]:
df.tail()

The CSV file contains data from Google from January 1, 2000 to December 1, 2024. <br />The data is based on monthly frequency and attempts to predict the future value of the "NDVI-mean" column. <br />Therefore, "NDVI-mean" is the target column here

View the shape of the data

In [ ]:
df.shape

Split training and testing. We cannot provide data here because it must be sequential in the time series.

In [ ]:
test_split=round(len(df)*0.20)
print(test_split)
df_for_training=df[:-53]
df_for_testing=df[-53:]
print(df_for_training.shape)
print(df_for_testing.shape)

It can be noted that the ranges of various factors are not consistent, so to avoid prediction errors, MinMaxScaler is first used to scale the data. (StandardScaler can also be used)

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
df_for_training_scaled = scaler.fit_transform(df_for_training)
df_for_testing_scaled=scaler.transform(df_for_testing)
df_for_training_scaled

Split the data into X and Y

In [ ]:
def createXY(dataset,n_past):
    dataX = []
    dataY = []
    for i in range(n_past, len(dataset)):
            dataX.append(dataset[i - n_past:i, 0:dataset.shape[1]])
            dataY.append(dataset[i,0])
    return np.array(dataX),np.array(dataY)
trainX,trainY=createXY(df_for_training_scaled,3)
testX,testY=createXY(df_for_testing_scaled,3)

N_past is the number of steps we will look at in the past when predicting the next target value<br/>Here, using 3 means that we will use the past 3 values (including all characteristics of the target column) to predict the fourth target value<br/>Therefore, in trainX we will have all the feature values, while in trainY we only have the target value<br/>Let's decompose each part of the for loop<br/>For training, dataset=df_for_training_Scaled, n_past=3<br/>When i= 3:<br />data_X.addend (df_for_training_scaled[i - n_past:i, 0:df_for_training.shape[1]])<br /> The range starting from n_past is 3, so the first data range will be - [3-3, 5, 0:5], which is equivalent to [0:3, 0:5]. Therefore, in the dataX list, the df_for_trainingscaled [0:3, 0:5] array will appear for the first time. Now, dataY.append (df_for_trainingscaled [i, 0])<br/>i=3, so it will only take the NDVI-bean starting from the third row (because in prediction, we only need the NDVI-bean column, so the column range is only 0, indicating an open column).<br/>The first time storing df_for_teans in the dataY list The rainingscaled [5,0] value is<br/>, so the first 3 rows containing 5 columns are stored in dataX, and only the 4th row of the open column is stored in dataY. Then we convert the dataX and dataY lists into arrays, which are trained in LSTM in array format

In [ ]:
print("trainX Shape-- ",trainX.shape)
print("trainY Shape-- ",trainY.shape)

In [ ]:
print("testX Shape-- ",testX.shape)
print("testY Shape-- ",testY.shape)

In the trainY of each array, we use the next target value to train the model

In [ ]:
print("trainX[0]-- \n",trainX[0])
print("trainY[0]-- ",trainY[0])

If you look at the value of trainX [1], you will find that it is the same as the data in trainX [0] (except for the first column), because we will see the first three to predict the fourth column, and after the first prediction, it will automatically move to the second column and take the next three values to predict the next target value.<br/>Use girdsearchCV to make some hyperparameter adjustments to find the underlying model

In [ ]:
def build_model(optimizer):
    grid_model = Sequential()
    grid_model.add(LSTM(units=64,return_sequences=True,input_shape=(3,9),activation='relu'))
    grid_model.add(LSTM(units=32,return_sequences=True,activation='relu'))
    grid_model.add(LSTM(units=16,return_sequences=True,activation='relu'))
    grid_model.add(LSTM(units=8,activation='relu'))
    grid_model.add(Dropout(0.5))
    grid_model.add(Dense(1))

    grid_model.compile(loss = 'mse',optimizer = optimizer, metrics=['accuracy'])
    return grid_model

grid_model = KerasRegressor(build_fn=build_model,verbose=1)
parameters = {'batch_size' : [4,8,16,32,64,128],
              'epochs' : [5,10,20,50,100,150,200],
              'optimizer' : ['SGD', 'adam', 'Momentum', 'Adagrad', 'RMSProp'] }

grid_search  = GridSearchCV(estimator = grid_model,
                            param_grid = parameters,
                            cv = 2)

In the first LSTM layer, the input shape is seen as (3,5). It comes from the shape of trainX. (trainX. shape [1], trainX. shape [2]) → (3,5)<br/>Fit the model to the trainX and trainY data

In [ ]:
grid_search_results = grid_search.fit(trainX, trainY, validation_data=(testX, testY),verbose=1)

In [ ]:
history = grid_search_results.best_estimator_.model.history

In [ ]:
print(history.history.keys())

In [ ]:
import time

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()
time.sleep(1)
plt.close()

Check the optimal parameters of the model.<br/>Save the best model in the my_madel variable.

In [ ]:
grid_search_results.best_params_

my_model=grid_search_results.best_estimator_.model

In [ ]:
prediction=my_model.predict(testX)
print("prediction\n", prediction)
print("\nPrediction Shape-",prediction.shape)

The length of testY and prediction is the same. Now we can compare testY with predictions.<br/>
But the data was scaled from the beginning, so some inverse scaling process must be done first.

In [ ]:
prediction_copies_array = np.repeat(prediction,9, axis=-1)
prediction_copies_array.shape

In [ ]:
pred=scaler.inverse_transform(np.reshape(prediction_copies_array,(len(prediction),9)))[:,0]

Compare this pred value with testY, but testY is also scaled proportionally, and the same code as above needs to be used for inverse transformation

In [ ]:
original_copies_array = np.repeat(testY,9, axis=-1)
original=scaler.inverse_transform(np.reshape(original_copies_array,(len(testY),9)))[:,0]

In [ ]:
print("Pred Values-- " ,pred)
print("\nOriginal Values-- " ,original)

Draw a graph to compare pred and raw data

In [ ]:
plt.plot(original, color = 'deepskyblue', label = 'Real NDVI')
plt.plot(pred, color = 'red', label = 'Predicted NDVI')
plt.title('NDVI Prediction', fontsize=16, fontname="Times New Roman")
plt.xlabel('Time', fontsize=14, fontname="Times New Roman")
plt.ylabel('NDVI', fontsize=14, fontname="Times New Roman")
plt.grid(True, which="major", axis="both", alpha=0.15, color="gray")
plt.legend(prop={"size": 12, "family": "Times New Roman"})
plt.ylim(0)
plt.xlim(-1,51)
plt.xticks(fontname="Times New Roman", fontsize=14)
plt.yticks(fontname="Times New Roman", fontsize=14)
plt.savefig('E:/1.jpg')
plt.show()


In [ ]:
f = open("E:/1.csv","w")
pd.set_option('display.max_columns', None)   
pd.set_option('display.max_rows', None)  
print(pred,file=f)
f.close()

In [ ]:
np.savetxt("E:/1.csv", pred, delimiter=",", comments="")

In [ ]:
def mda(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Directional Accuracy """
    return np.mean((np.sign(actual[1:] - actual[:-1]) == np.sign(predicted[1:] - predicted[:-1])).astype(int))

def mean_absolute_percentage_error(original, pred): 
    y_true, y_pred = np.array(original), np.array(original)
    return np.mean(np.abs((original - pred) / original)) * 100

In [ ]:
r2 = r2_score(original, pred)
print('R2 value of the LSTM Model is:', r2)

MAE_lstm = mean_absolute_error(original, pred)
print('MAE value of the LSTM Model is:', MAE_lstm)

MDA_lstm = mda(original, pred)
print('MDA value of the LSTM Model is:', MDA_lstm)

MAPE_lstm = mean_absolute_percentage_error(original, pred)
print('MAPE value of the LSTM Model is:', MAPE_lstm)

RMSE_lstm = sqrt(mean_squared_error(original,pred, squared=False))
print('RMSE value of the LSTM Model is:', RMSE_lstm)

MSE_lstm = mean_squared_error(original,pred)
print('MSE value of the LSTM Model is:', MSE_lstm)

EVS_lstm = explained_variance_score(original, pred)
print('EVS score of the LSTM Model is:', EVS_lstm)

In [ ]:
import csv

data = [[r2], [MAE_lstm], [MDA_lstm], [MAPE_lstm], [RMSE_lstm], [MSE_lstm], [EVS_lstm]]

with open('E:/data.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    for row in data:
        writer.writerow(row)


Predict future values<br/>Obtain the last 3 values we loaded at the beginning from the main df dataset.

In [ ]:
df_36_monthes_past=df.iloc[-36:,:]
df_36_monthes_past.tail()

In multivariate time series prediction, it is necessary to use different features to predict a single column, so when making predictions, we need to use feature values (excluding the target column) for upcoming predictions.

We need the upcoming 36 values from other columns here to predict the 'NDVI-mean' column.

In [ ]:
df_36_monthes_future=pd.read_csv("./data_pred.csv",parse_dates=["DATE"],index_col=[0])
df_36_monthes_future.tail()

Scale the data by removing the 'NDVI-mean' column and adding an NDVI-mean column with all values set to '0' before scaling it.

After scaling, replace the column value of "NDVI-mean" in future data with "nan"

Now add 36 old values and 36 new values (with the last 36 "open" values being nan)

In [ ]:
df_36_monthes_future["NDVI_mean"]=0
df_36_monthes_future=df_36_monthes_future[["NDVI_mean","temp_mean","precip_mean","ET0_mean","LST_mean"]]
old_scaled_array=scaler.transform(df_36_monthes_past)
new_scaled_array=scaler.transform(df_36_monthes_future)
new_scaled_df=pd.DataFrame(new_scaled_array)
new_scaled_df.iloc[:,0]=np.nan
full_df=pd.concat([pd.DataFrame(old_scaled_array),new_scaled_df]).reset_index().drop(["index"],axis=1)

In [ ]:
full_df_scaled_array=full_df.values
all_data=[]
time_step=36
for i in range(time_step,len(full_df_scaled_array)):
    data_x=[]
    data_x.append(full_df_scaled_array[i-time_step :i , 0:full_df_scaled_array.shape[1]])
    data_x=np.array(data_x)
    prediction=my_model.predict(data_x)
    all_data.append(prediction)
    full_df.iloc[i,0]=prediction

For the first prediction, there are the previous three values, and when the for loop runs for the first time, it checks the first three values and predicts the fourth 'NDVI-mean' data.

When the second for loop attempts to run, it will skip the first line and attempt to obtain the next 3 values [1:3]. There will be an error here because the last row of the Open column is' nan ', so it is necessary to replace' nan 'with a prediction every time.

Finally, it is necessary to perform inverse transformation on the prediction

In [ ]:
new_array=np.array(all_data)
new_array=new_array.reshape(-1,1)
prediction_copies_array = np.repeat(new_array,5, axis=-1)
y_pred_future_36_monthes = scaler.inverse_transform(np.reshape(prediction_copies_array,(len(new_array),5)))[:,0]
print(y_pred_future_36_monthes)